# Alia: Qwen3.6-35B Q4 on a Colab T4

Measures Qwen3.6-35B-A3B UD-Q4_K_XL (22.4 GB) on a T4 (16 GB VRAM) with the llama.cpp build Alia pins (b11639) at
the 32,768 context of the `rtx12-16gb-qwen36-35b-q4` profile.

**Variant, not the exact profile:** the profile needs 30 GB RAM, a standard Colab runtime has 12.7 GB. The run adds
`--no-repack --no-host` so expert weights that do not fit in VRAM stay memory-mapped from disk.

**Run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
Every step prints a line at least every 20 seconds. The last cell disconnects the runtime so it stops using
compute units; the output stays on the page.

## 1. Setup: llama.cpp, model (22.4 GB), Alia

In [ ]:
import glob, json, os, re, select, shutil, subprocess, sys, time

def sh(cmd):
    """Run a shell command and show its output live."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()

def ram():
    m = {l.split(":")[0]: int(l.split()[1]) for l in open("/proc/meminfo")}
    return f"RAM {(m['MemTotal'] - m['MemAvailable']) / 1e6:.1f}/{m['MemTotal'] / 1e6:.1f} GB"

def vram():
    return subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()

R = "https://github.com/ggml-org/llama.cpp/releases/download/b11639"
F = "/content/models/Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf"
SIZE = 22360456160
URL = "https://huggingface.co/unsloth/Qwen3.6-35B-A3B-GGUF/resolve/main/Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf"
SETUP_OK = False
try:
    assert shutil.which("nvidia-smi"), "no GPU: Runtime -> Change runtime type -> T4 GPU"
    # Library path: drop /content/llama entries (an earlier version of this cell put them there and the kernel
    # keeps them) and make sure the driver libs (libcuda, libnvidia-ml) are on it, wherever Colab mounts them.
    lib_paths = [d for d in os.environ.get("LD_LIBRARY_PATH", "").split(":") if d and not d.startswith("/content/llama")]
    for lib in ["libcuda.so.1", "libnvidia-ml.so.1"]:
        hit = subprocess.run(f"find /usr /opt /lib -name {lib} 2>/dev/null | grep -v -e i386 -e /lib32/ | head -1",
                             shell=True, capture_output=True, text=True).stdout.strip()
        if hit and os.path.dirname(hit) not in lib_paths:
            lib_paths.insert(0, os.path.dirname(hit))
    os.environ["LD_LIBRARY_PATH"] = ":".join(lib_paths)
    print("LD_LIBRARY_PATH =", os.environ["LD_LIBRARY_PATH"], flush=True)
    sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; free -g | head -2; pkill -x llama-server || true")

    print("\n== llama.cpp b11639", flush=True)
    if not glob.glob("/content/llama/**/llama-server", recursive=True):
        for f in ["llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz", "cudart-llama-b11639-bin-ubuntu-cuda-12.8-x64.tar.gz"]:
            sh(f"cd /content && wget -q -c {R}/{f} && mkdir -p llama && tar xzf {f} -C llama")
    BIN = glob.glob("/content/llama/**/llama-server", recursive=True)[0]
    # CUDA runtime libs come in a separate tarball; next to llama-server they are found via its RUNPATH $ORIGIN
    sh(f"cp -n /content/llama/cudart-*/lib* {os.path.dirname(BIN)}/ 2>/dev/null; {BIN} --version 2>&1 | tail -2")
    devices = subprocess.run([BIN, "--list-devices"], capture_output=True, text=True)
    print(devices.stdout.strip(), flush=True)
    assert "CUDA0" in devices.stdout, "llama-server sees no GPU:\n" + devices.stdout + devices.stderr[-1500:]
    assert vram().split()[0].isdigit(), f"nvidia-smi failed: {vram()[:200]}"

    print("\n== model", flush=True)
    os.makedirs("/content/models", exist_ok=True)
    if not os.path.exists(F) or os.path.getsize(F) != SIZE:
        p = subprocess.Popen(["wget", "-q", "-c", "-O", F, URL])
        while p.poll() is None:
            time.sleep(15)
            have = os.path.getsize(F) if os.path.exists(F) else 0
            print(f"{time.strftime('%H:%M:%S')}  {have / 1e9:5.1f} / 22.4 GB  ({have * 100 // SIZE} %)  {ram()}", flush=True)
    assert os.path.getsize(F) == SIZE, "model download incomplete: run this cell again"
    print("model complete", flush=True)

    print("\n== Alia", flush=True)
    assert sh("cd /content && ([ -d alia ] || git clone -q --branch micro/r10 https://github.com/BEKO2210/alia) "
              "&& cd alia && git pull -q && git log --oneline -1") == 0, "git clone/pull failed"
    SETUP_OK = True
    print("\nSETUP OK")
except Exception as e:
    print("\nSETUP FAILED:", e)

## 2. Measure (3 reps at 4K, 16K and 28K tokens of context) and result

In [ ]:
NAME = "colab-t4-12-16gb-norepack"
MEASURE_OK = False
if not SETUP_OK:
    print("setup did not finish: run cell 1 again")
else:
    os.chdir("/content/alia")
    os.environ["LLAMA_SERVER"] = BIN
    LOG = f"bench/results/llama-server-{NAME}-fit.log"
    if os.path.exists(LOG):
        os.remove(LOG)
    started = time.time()
    cmd = [sys.executable, "-u", "bench/bench_llamacpp.py", "--gguf", F, "--name", NAME, "--configs", "fit",
           "--depths", "4096,16384,28000", "--reps", "3", "--ctx", "32768", "--load-timeout", "900",
           "--extra", "-b 2048 -ub 2048 --jinja --cache-ram 2048 --no-repack --no-host"]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    while p.poll() is None:
        if select.select([p.stdout], [], [], 20)[0]:
            line = p.stdout.readline()
            if line:
                print(line, end="", flush=True)
                continue
        last = ""
        if os.path.exists(LOG):
            with open(LOG, "rb") as fh:
                fh.seek(max(0, os.path.getsize(LOG) - 2000))
                tail = fh.read().decode(errors="replace").strip().splitlines()
                last = tail[-1][:90] if tail else ""
        print(f"[{(time.time() - started) / 60:4.1f} min] {ram()} | VRAM {vram()} | server: {last}", flush=True)
    for line in p.stdout:
        print(line, end="", flush=True)

    rows = []
    for f in sorted(glob.glob("bench/results/llamacpp-*.jsonl"), key=os.path.getmtime):
        rows += [r for r in map(json.loads, open(f)) if r.get("model") == NAME]
    print("\n================ RESULT (copy from here) ================")
    print(f"{'depth':>6} {'pp tok/s':>9} {'tg tok/s':>9} {'VRAM MiB':>9} {'RSS MiB':>8} {'load s':>7}")
    for r in rows:
        if "pp_tps" in r:
            print(f"{r['depth']:>6} {r['pp_tps']['median']:>9.0f} {r['tg_tps']['median']:>9.1f} "
                  f"{r['vram_peak_mib']:>9.0f} {r.get('rss_peak_mib', 0):>8} {r['load_s']:>7.0f}")
        else:
            print("ERROR", r.get("error", r))
    if os.path.exists(LOG):
        log = open(LOG, errors="replace").read().splitlines()
        print("\n--- server: offload / buffers")
        print("\n".join(l for l in log if re.search(r"offload|model buffer size|KV buffer size|fit", l))[-2500:])
        print("\n--- server log tail")
        print("\n".join(log[-15:]))
    print("\n--- raw JSONL")
    for r in rows:
        print(json.dumps(r))
    MEASURE_OK = any("pp_tps" in r for r in rows)

## 3. Stop the runtime after a successful run (stops compute-unit use)

In [ ]:
if MEASURE_OK:
    from google.colab import runtime
    print("measurement done: disconnecting the runtime; the output above stays on this page")
    runtime.unassign()
else:
    print("NO RESULT. The runtime keeps running so the 22 GB model need not be downloaded again.")
    print("Re-run cell 2 after a fix, or stop it now: Runtime -> Disconnect and delete runtime.")